# TSM 動能 × 趨勢 × 量先價後延遲回應：Colab 可重現 notebook

這份 notebook 把既有 Study 的策略邏輯整理成可以公開閱讀、上傳 CSV 並離線重跑的示例。它不是正式 Evaluation，也不會自動下載 Yahoo 資料、連線券商或送出任何委託。

## Study 身分與研究目的

- Study ID：`tsm-momentum-trend-volume-response-lag--v001`
- 研究標的：TSM 的日線、已調整 OHLCV。
- 目的：檢查訊號日前已完成的量能—下一 session 收盤到收盤報酬配對，是否能在上升趨勢與價格加速成立時，提供額外的動能承接篩選。

## 假說與策略差異

在訊號日 `t`，先取前五個已完成的配對：每個觀測日 `d` 的成交量，對應 `d→d+1` 的收盤到收盤報酬。五日成交量加權後續報酬必須至少為 `-0.02`，而且要比同五筆報酬的未加權算術平均高至少 `0.0005`。這個跨日條件再和以下當日確認一起使用：五日 SMA 趨勢斜率比至少 `0.95`、收盤不低於 SMA 的 `0.90`、二期 RSI 不高於 `70`、收盤高於前收、單日價格加速至少 `2%`，以及前五日平均量比至少 `1.05`。

候選（candidate）啟用上述延遲回應條件；baseline 保留相同的價格條件、五日平均量比、進出場、成本、風險與冷卻規則，只關閉這個跨日量—下一日報酬傳導條件。因此兩者的差異可歸因於是否使用延遲回應篩選，而不是不同的持倉或風控。

本 notebook 的 synthetic 輸出只是確認程式能運作的 smoke test；上傳資料的重跑是 reproducer，不是既有 Development evidence，也不能外推正式 Evaluation、Terminal 或任何未在本 notebook 中執行的研究結論。正式 Study 的 candidate 只到 Development，且不得在這裡調參、重跑或改寫既有證據。

## v004 policy 與執行規則摘要

本頁的 digest 是 Study 建立時固定的 v004 binding，供讀者確認 notebook 對應哪一版規格；程式本身不會在執行時重新解析 repository。

| 項目 | 固定內容 |
| --- | --- |
| Workflow | `strategy-forward-replication-research` v004；workflow digest `62779bce18802e32ab314b6d74e8fc6f2da9fac03d1ee85a6416acc5553c67e4` |
| Workflow reference | `7b13d4d7e6448c9858215d9ef7e2e62fbd7fe0f40502e95a778a091008b20b47` |
| Release manifest | `ea04558c1473f9c6db7e9707846694147c6c6f254498af2f18729a1e4ef1fa84` |
| Policy set | `c86066b33119366a3172f475ff75f8813ba4b7545571894edfe581afabe32215` |
| Policy components | `canonical-execution--v001`、`paper-proposal-orders--v001`、`portfolio-risk--v001`、`us-equity-market--v002` |
| Source bundle | `06ed3b3bdb9c596df4c213d2fb657b368de9950d3ddbe1282ccb408a3639c426` |
| Preregistration | `8018ba3fff138c288c0e4d863ca75fc45ec30d3f219c88743d6ae035f94ba7d9` |

### 資料與 session

Study 的來源約束是 Yahoo 的 `auto_adjusted`、日線資料與 XNYS session。也就是說，拆股與股利調整應已反映在上傳的 OHLCV；session 是美國交易所實際完成的交易日，而不是把週末或任意時間戳當成交易日。本 notebook 不下載 Yahoo；使用者必須自行提供已整理成 XNYS session-clean 的 CSV。程式會驗證日期可解析、去除時間部分後不重複、嚴格遞增、不含週末，以及 OHLCV 的數值關係，但不靠外部日曆猜測缺少的交易所假日。

### 執行、風控與成本的實際意義

- **completed-session-close → next-session-open**：訊號只能在當日收盤完成後形成，下一個 XNYS session 的開盤價才是進場價格；不能用訊號日收盤價假裝成交。
- **2% risk budget**：每次用進場前資產的 2% 估算到停損價、含進出成本的最大風險，再取同時符合現金上限的最大整數股數；不可借錢，也不會因為看好而加碼。
- **-4% stop、+4% target**：停損和停利水準都從原始進場 open 計算。停損是 GTC stop-market，停利是每個持倉 session 重新掛出的當日 limit；若開盤已跳過水準，直接以開盤處理。
- **最多 10 個完整 session**：若十個完整持倉 session 都沒有先觸發停損或停利，下一個 open 以 time exit 離場。
- **退出後 5-session cooldown**：平倉後要等五個完成的 session 步驟，才允許新的訊號；持倉不可重疊。
- **adverse-stop-first**：同一根日線同時碰到停損與停利時，採較保守的停損先發生；這是日線資料無法知道盤中順序時的固定解釋。
- **base 成本**：每邊手續費 `1 bps`、每邊滑價 `5 bps`。**stress 成本**：每邊手續費 `2 bps`、每邊滑價 `20 bps`；stress 不是另一個策略，而是檢查較差成交條件下結果是否仍可讀。
- **no real orders / paper proposal**：這裡只做離線回測和交易 ledger；policy 禁止建立 broker order，paper proposal 也不可被當成可執行的真實交易。

## 1. Notebook 設定與資料切換

預設 `USE_UPLOADED_DATA=False`，所以 Colab、Jupyter 或 nbclient 的非互動執行都會使用固定 seed 的 synthetic smoke fixture，不會卡在 `files.upload()`。在 Colab 中要使用自己的 CSV 時，將開關改成 `True` 後重新執行這個 cell；該 cell 才會顯示上傳對話框。

In [ ]:
import math
from dataclasses import dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

# 預設為 False，保證沒有上傳檔案時也能完整執行。
USE_UPLOADED_DATA = False
UPLOADED_FILENAME: str | None = None
SIGNAL_START: str | None = None
SIGNAL_END: str | None = None
INITIAL_CASH = 100_000.0
SMOKE_SEED = 20260922

print('設定完成：USE_UPLOADED_DATA =', USE_UPLOADED_DATA)

## 2. 自包含的策略規格、成本與輸出結構

下列 dataclass 是 notebook 內的最小實作，不 import repository 的 `trading_2026_2` package。名稱和數值對應 Study implementation contract；candidate 與 baseline 共用同一份執行設定。

In [ ]:
@dataclass(frozen=True)
class CostModel:
    slippage_bps: float
    fee_bps: float


@dataclass(frozen=True)
class StrategySpec:
    sma_lookback: int = 20
    sma_min_periods: int = 20
    mean_reversion_min: float = -0.10
    rsi_lookback: int = 2
    rsi_min_periods: int = 2
    rsi_max: float = 70.0
    volume_lookback: int = 20
    volume_average_min_periods: int = 20
    volume_lead_window: int = 5
    volume_lead_min_periods: int = 5
    volume_spike_ratio: float = 1.05
    volume_forward_response_enabled: bool = True
    volume_forward_return_min: float = -0.02
    volume_forward_advantage_min: float = 0.0005
    volume_lead_enabled: bool = True
    require_close_above_prior_close: bool = True
    trend_slope_lookback: int = 5
    trend_slope_floor: float = 0.95
    price_to_sma_floor: float = 0.90
    price_acceleration_lookback: int = 1
    price_acceleration_min: float = 0.02
    cooldown_sessions: int = 5
    target_return: float = 0.04
    stop_return: float = -0.04
    holding_sessions: int = 10
    fold_warmup_sessions: int = 25
    initial_cash: float = INITIAL_CASH
    risk_fraction: float = 0.02

    def with_changes(self, **changes: object) -> 'StrategySpec':
        return replace(self, **changes)


BASE_COST = CostModel(slippage_bps=5.0, fee_bps=1.0)
STRESS_COST = CostModel(slippage_bps=20.0, fee_bps=2.0)
DEFAULT_SPEC = StrategySpec()
BASELINE_SPEC = DEFAULT_SPEC.with_changes(volume_forward_response_enabled=False)


@dataclass(frozen=True)
class Trade:
    signal_session: pd.Timestamp
    entry_session: pd.Timestamp
    exit_session: pd.Timestamp
    raw_entry_price: float
    raw_exit_price: float
    executed_entry_price: float
    executed_exit_price: float
    shares: int
    fees: float
    pnl: float
    exit_reason: str
    held_sessions: int


@dataclass(frozen=True)
class BacktestResult:
    trades: tuple[Trade, ...]
    accepted_signal_sessions: tuple[pd.Timestamp, ...]
    ending_cash: float

print('規格已載入：base=', BASE_COST, 'stress=', STRESS_COST)

## 3. CSV 與 XNYS session-clean 驗證

CSV 預期包含 `Date` 或 `Datetime`，以及 `Open`、`High`、`Low`、`Close`、`Volume`。額外欄位會被忽略。驗證會阻擋日期缺失、重複、倒序、週末、非數值、非有限值、負價格、負成交量及不可能的高低價關係。交易所假日不會被當成錯誤，因為 notebook 不內嵌外部交易所日曆；上傳者仍必須自行保證資料已是 XNYS session-clean。

In [ ]:
def _normalise_date_series(values: pd.Series) -> pd.DatetimeIndex:
    parsed = pd.to_datetime(values, errors='coerce')
    if parsed.isna().any():
        raise ValueError('Date/Datetime 含有無法解析的值')
    timezone = getattr(parsed.dt, 'tz', None)
    if timezone is not None:
        parsed = parsed.dt.tz_convert('America/New_York').dt.tz_localize(None)
    return pd.DatetimeIndex(parsed.dt.normalize())


def validate_ohlcv(frame: pd.DataFrame) -> pd.DataFrame:
    required = ['Open', 'High', 'Low', 'Close', 'Volume']
    missing = [name for name in required if name not in frame.columns]
    if missing:
        raise ValueError('缺少必要欄位: ' + ', '.join(missing))
    data = frame.loc[:, required].copy()
    if not isinstance(data.index, pd.DatetimeIndex):
        raise ValueError('內部資料 index 必須是 DatetimeIndex')
    index = data.index
    if index.tz is not None:
        index = index.tz_convert('America/New_York').tz_localize(None)
    index = index.normalize()
    data.index = index
    data.index.name = 'Date'
    if len(data) == 0:
        raise ValueError('OHLCV 不得為空')
    if data.index.has_duplicates:
        raise ValueError('session 日期不得重複')
    if not data.index.is_monotonic_increasing:
        raise ValueError('session 日期必須嚴格遞增')
    if (data.index.weekday >= 5).any():
        raise ValueError('session 日期不得包含週末；請提供 XNYS session-clean 資料')
    for name in required:
        data[name] = pd.to_numeric(data[name], errors='coerce')
    values = data.to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('OHLCV 不得包含缺值或無限值')
    if (data[['Open', 'High', 'Low', 'Close']] <= 0).any().any():
        raise ValueError('OHLC 價格必須大於 0')
    if (data['Volume'] < 0).any():
        raise ValueError('Volume 不得小於 0')
    row_high = data[['Open', 'Close', 'Low']].max(axis=1)
    row_low = data[['Open', 'Close', 'High']].min(axis=1)
    if (data['High'] < row_high).any() or (data['Low'] > row_low).any():
        raise ValueError('OHLC 高低價關係不合法')
    return data


def prepare_ohlcv(frame: pd.DataFrame) -> pd.DataFrame:
    date_columns = [name for name in ['Date', 'Datetime'] if name in frame.columns]
    if len(date_columns) > 1:
        raise ValueError('請只保留 Date 或 Datetime 其中一個日期欄位')
    data = frame.copy()
    if date_columns:
        data.index = _normalise_date_series(data[date_columns[0]])
    elif not isinstance(data.index, pd.DatetimeIndex):
        raise ValueError('CSV 必須包含 Date 或 Datetime 欄位')
    return validate_ohlcv(data)

print('資料驗證函式已載入')

## 4. 指標與 candidate / baseline signal

關鍵防 look-ahead 規則是：訊號日 `t` 的延遲回應只看 `t-6` 到 `t-2` 的五筆成交量，並配對到 `t-6→t-5` 到 `t-2→t-1` 的五筆已完成收盤報酬。`shift(2)` 讓最後一筆配對在訊號日開始前已完成；訊號日之後的 Close 不會進入 forward-response 欄位。

In [ ]:
def simple_rsi(close: pd.Series, length: int, min_periods: int) -> pd.Series:
    change = close.diff()
    gain = change.clip(lower=0).rolling(length, min_periods=min_periods).mean()
    loss = (-change.clip(upper=0)).rolling(length, min_periods=min_periods).mean()
    with np.errstate(divide='ignore', invalid='ignore'):
        rsi = 100.0 - 100.0 / (1.0 + gain / loss)
    ready = gain.notna() & loss.notna()
    rsi = rsi.where(ready, np.nan)
    rsi = rsi.mask(ready & (gain == 0) & (loss == 0), 50.0)
    rsi = rsi.mask(ready & (loss == 0) & (gain > 0), 100.0)
    rsi = rsi.mask(ready & (gain == 0) & (loss > 0), 0.0)
    return rsi


def required_history_sessions(spec: StrategySpec) -> int:
    return max(
        spec.sma_min_periods - 1,
        spec.rsi_min_periods,
        spec.volume_average_min_periods + spec.volume_lead_window,
        spec.sma_min_periods - 1 + spec.trend_slope_lookback,
        spec.volume_lead_window + 2,
    )


def indicators(bars: pd.DataFrame, spec: StrategySpec = DEFAULT_SPEC) -> pd.DataFrame:
    clean = validate_ohlcv(bars)
    close = clean['Close']
    volume = clean['Volume']
    sma = close.rolling(spec.sma_lookback, min_periods=spec.sma_min_periods).mean()
    rsi = simple_rsi(close, spec.rsi_lookback, spec.rsi_min_periods)
    prior_close = close.shift(1)
    prior_volume_average = volume.shift(1).rolling(
        spec.volume_lookback, min_periods=spec.volume_average_min_periods
    ).mean()
    volume_spike_ratio = volume / prior_volume_average
    prior_volume_ratio = volume_spike_ratio.shift(1).rolling(
        spec.volume_lead_window, min_periods=spec.volume_lead_min_periods
    ).mean()

    # d 日成交量只和 d→d+1 的收盤到收盤報酬配對；shift(2) 排除 t-1 與 t。
    forward_return = close.shift(-1) / close - 1.0
    paired_volume = volume.shift(2).rolling(
        spec.volume_lead_window, min_periods=spec.volume_lead_min_periods
    ).sum()
    paired_volume_return = (volume * forward_return).shift(2).rolling(
        spec.volume_lead_window, min_periods=spec.volume_lead_min_periods
    ).sum()
    prior_volume_forward_return = paired_volume_return / paired_volume
    prior_unweighted_forward_return = forward_return.shift(2).rolling(
        spec.volume_lead_window, min_periods=spec.volume_lead_min_periods
    ).mean()
    prior_volume_forward_advantage = (
        prior_volume_forward_return - prior_unweighted_forward_return
    )
    response_ready = (
        prior_volume_ratio.notna()
        & prior_volume_forward_return.notna()
        & prior_unweighted_forward_return.notna()
        & prior_volume_forward_advantage.notna()
    )
    prior_volume_ratio = prior_volume_ratio.where(response_ready)
    prior_volume_forward_return = prior_volume_forward_return.where(response_ready)
    prior_unweighted_forward_return = prior_unweighted_forward_return.where(response_ready)
    prior_volume_forward_advantage = prior_volume_forward_advantage.where(response_ready)

    trend_slope_ratio = sma / sma.shift(spec.trend_slope_lookback)
    price_to_sma = close / sma
    price_acceleration = close / close.shift(spec.price_acceleration_lookback) - 1.0
    close_above_prior_close = close > prior_close
    trend_regime = trend_slope_ratio >= spec.trend_slope_floor
    volume_ratio_lead = prior_volume_ratio >= spec.volume_spike_ratio
    if spec.volume_forward_response_enabled:
        volume_forward_response_lead = (
            (prior_volume_forward_return >= spec.volume_forward_return_min)
            & (prior_volume_forward_advantage >= spec.volume_forward_advantage_min)
        )
    else:
        volume_forward_response_lead = pd.Series(True, index=clean.index)
    volume_lead = volume_ratio_lead & volume_forward_response_lead
    momentum_signal = (
        trend_regime
        & (price_to_sma >= spec.price_to_sma_floor)
        & (price_acceleration >= spec.price_acceleration_min)
        & (rsi <= spec.rsi_max)
        & rsi.notna()
        & (close_above_prior_close | ~spec.require_close_above_prior_close)
        & ((sma - close) / sma >= spec.mean_reversion_min)
    )
    raw_signal = (momentum_signal & volume_lead).fillna(False).astype(bool)

    result = clean.copy()
    result['sma_20'] = sma
    result['rsi_2'] = rsi
    result['prior_volume_average'] = prior_volume_average
    result['volume_spike_ratio'] = volume_spike_ratio
    result['prior_volume_ratio'] = prior_volume_ratio
    result['prior_volume_forward_return'] = prior_volume_forward_return
    result['prior_unweighted_forward_return'] = prior_unweighted_forward_return
    result['prior_volume_forward_advantage'] = prior_volume_forward_advantage
    result['trend_slope_ratio'] = trend_slope_ratio
    result['price_to_sma'] = price_to_sma
    result['price_acceleration'] = price_acceleration
    result['prior_close'] = prior_close
    result['close_above_prior_close'] = close_above_prior_close
    result['trend_regime'] = trend_regime
    result['volume_ratio_lead'] = volume_ratio_lead
    result['volume_forward_response_lead'] = volume_forward_response_lead
    result['volume_lead'] = volume_lead
    result['raw_momentum_signal'] = momentum_signal
    result['raw_signal'] = raw_signal
    return result

print('指標函式已載入；candidate 與 baseline 只在 delayed response filter 上不同')

## 5. next-open execution、風險預算、退出與 metrics

這裡把 implementation contract 的交易生命週期完整放在 notebook：訊號後下一個 open 成交、成本內含的 stop 風險 sizing、gap fill、adverse-stop-first、十個完整 session time exit、五 session cooldown，以及每筆交易 ledger。

In [ ]:
def entry_fill(raw_open: float, cost: CostModel) -> float:
    return raw_open * (1.0 + cost.slippage_bps / 10_000.0)


def exit_fill(raw_price: float, cost: CostModel) -> float:
    return raw_price * (1.0 - cost.slippage_bps / 10_000.0)


def risk_budget_shares(
    cash: float, raw_entry: float, *, stop_return: float, risk_fraction: float, cost: CostModel
) -> int:
    executed_entry = entry_fill(raw_entry, cost)
    executed_stop = exit_fill(raw_entry * (1.0 + stop_return), cost)
    entry_cash_per_share = executed_entry * (1.0 + cost.fee_bps / 10_000.0)
    stop_proceeds_per_share = executed_stop * (1.0 - cost.fee_bps / 10_000.0)
    modeled_loss_per_share = entry_cash_per_share - stop_proceeds_per_share
    if modeled_loss_per_share <= 0:
        raise ValueError('成本內含的 stop 每股損失必須大於 0')
    affordable = math.floor(cash / entry_cash_per_share)
    risk_limited = math.floor((cash * risk_fraction) / modeled_loss_per_share)
    return max(0, min(affordable, risk_limited))


def intraday_exit(bar: pd.Series, target: float, stop: float) -> tuple[float, str] | None:
    if bar['Open'] <= stop:
        return float(bar['Open']), 'stop-gap'
    if bar['Open'] >= target:
        return float(bar['Open']), 'target-gap'
    stop_hit = bar['Low'] <= stop
    target_hit = bar['High'] >= target
    if stop_hit and target_hit:
        return stop, 'stop-same-session'
    if stop_hit:
        return stop, 'stop'
    if target_hit:
        return target, 'target'
    return None


def backtest(
    bars: pd.DataFrame,
    *,
    spec: StrategySpec = DEFAULT_SPEC,
    cost: CostModel = BASE_COST,
    signal_start: str | None = None,
    signal_end: str | None = None,
) -> BacktestResult:
    if spec.cooldown_sessions < 0 or spec.holding_sessions <= 0:
        raise ValueError('cooldown 與持有期設定不合法')
    frame = indicators(bars, spec)
    cash = float(spec.initial_cash)
    trades: list[Trade] = []
    accepted: list[pd.Timestamp] = []
    pending_signal_index: int | None = None
    last_exit_index: int | None = None
    position: dict[str, object] | None = None
    time_exit_index: int | None = None
    first_signal_index = required_history_sessions(spec)
    start = pd.Timestamp(signal_start).normalize() if signal_start is not None else None
    end = pd.Timestamp(signal_end).normalize() if signal_end is not None else None
    if start is not None and end is not None and start > end:
        raise ValueError('signal_start 不得晚於 signal_end')

    def close_position(
        session: pd.Timestamp, raw_exit: float, exit_reason: str, held_sessions: int, state: dict[str, object]
    ) -> None:
        nonlocal cash, position, time_exit_index, last_exit_index
        executed_exit = exit_fill(raw_exit, cost)
        shares = int(state['shares'])
        exit_fee = shares * executed_exit * cost.fee_bps / 10_000.0
        cash += shares * executed_exit - exit_fee
        total_fees = float(state['entry_fee']) + exit_fee
        pnl = cash - float(state['cash_before_entry'])
        trades.append(
            Trade(
                signal_session=state['signal_session'],
                entry_session=state['entry_session'],
                exit_session=session,
                raw_entry_price=float(state['raw_entry']),
                raw_exit_price=raw_exit,
                executed_entry_price=float(state['executed_entry']),
                executed_exit_price=executed_exit,
                shares=shares,
                fees=total_fees,
                pnl=pnl,
                exit_reason=exit_reason,
                held_sessions=held_sessions,
            )
        )
        position = None
        time_exit_index = None
        last_exit_index = frame.index.get_loc(session)

    for index, (session, bar) in enumerate(frame.iterrows()):
        if position is not None and time_exit_index == index:
            close_position(session, float(bar['Open']), 'time', spec.holding_sessions, position)

        if pending_signal_index is not None and pending_signal_index + 1 == index:
            raw_entry = float(bar['Open'])
            executed_entry = entry_fill(raw_entry, cost)
            cash_before_entry = cash
            shares = risk_budget_shares(
                cash, raw_entry, stop_return=spec.stop_return, risk_fraction=spec.risk_fraction, cost=cost
            )
            if shares > 0:
                entry_fee = shares * executed_entry * cost.fee_bps / 10_000.0
                cash -= shares * executed_entry + entry_fee
                position = {
                    'signal_session': frame.index[pending_signal_index],
                    'entry_session': session,
                    'raw_entry': raw_entry,
                    'executed_entry': executed_entry,
                    'entry_fee': entry_fee,
                    'cash_before_entry': cash_before_entry,
                    'shares': shares,
                    'target': raw_entry * (1.0 + spec.target_return),
                    'stop': raw_entry * (1.0 + spec.stop_return),
                    'held_sessions': 0,
                }
                time_exit_index = index + spec.holding_sessions
            pending_signal_index = None

        if position is not None:
            exit_match = intraday_exit(bar, float(position['target']), float(position['stop']))
            if exit_match is not None:
                raw_exit, exit_reason = exit_match
                close_position(session, raw_exit, exit_reason, int(position['held_sessions']) + 1, position)
            else:
                position['held_sessions'] = int(position['held_sessions']) + 1

        enough_time_to_exit = index + spec.holding_sessions + 1 < len(frame)
        lifecycle_within_end = (
            end is None
            or not enough_time_to_exit
            or frame.index[index + spec.holding_sessions + 1] <= end
        )
        cooldown_ready = (
            last_exit_index is None
            or index - last_exit_index >= spec.cooldown_sessions
        )
        date_is_eligible = (start is None or session >= start) and (end is None or session <= end)
        if (
            index >= first_signal_index
            and date_is_eligible
            and enough_time_to_exit
            and lifecycle_within_end
            and position is None
            and pending_signal_index is None
            and cooldown_ready
            and bool(bar['raw_signal'])
        ):
            accepted.append(session)
            pending_signal_index = index

    return BacktestResult(tuple(trades), tuple(accepted), cash)


def qualification_metrics(result: BacktestResult, initial_cash: float = INITIAL_CASH) -> dict[str, float | int]:
    pnls = [trade.pnl for trade in result.trades]
    gross_profit = sum(value for value in pnls if value > 0)
    gross_loss = -sum(value for value in pnls if value < 0)
    profit_factor = gross_profit / gross_loss if gross_loss else (math.inf if gross_profit else 0.0)
    equity = initial_cash
    peak = initial_cash
    maximum_drawdown = 0.0
    for pnl in pnls:
        equity += pnl
        peak = max(peak, equity)
        maximum_drawdown = max(maximum_drawdown, (peak - equity) / peak)
    return {
        'completed_trades': len(result.trades),
        'traded_years': len({trade.signal_session.year for trade in result.trades}),
        'return': sum(pnls) / initial_cash,
        'profit_factor': profit_factor,
        'maximum_drawdown': maximum_drawdown,
    }


def equity_curve(result: BacktestResult, initial_cash: float = INITIAL_CASH) -> pd.DataFrame:
    rows = [{'Date': pd.Timestamp('1900-01-01'), 'Equity': initial_cash}]
    equity = initial_cash
    for trade in sorted(result.trades, key=lambda item: item.exit_session):
        equity += trade.pnl
        rows.append({'Date': trade.exit_session, 'Equity': equity})
    curve = pd.DataFrame(rows)
    curve['Date'] = pd.to_datetime(curve['Date'])
    return curve


def ledger_for_result(model: str, scenario: str, result: BacktestResult) -> list[dict[str, object]]:
    return [
        {
            'model': model,
            'scenario': scenario,
            'signal_session': trade.signal_session.date().isoformat(),
            'entry_session': trade.entry_session.date().isoformat(),
            'exit_session': trade.exit_session.date().isoformat(),
            'exit_reason': trade.exit_reason,
            'held_sessions': trade.held_sessions,
            'shares': trade.shares,
            'raw_entry_price': trade.raw_entry_price,
            'raw_exit_price': trade.raw_exit_price,
            'executed_entry_price': trade.executed_entry_price,
            'executed_exit_price': trade.executed_exit_price,
            'fees': trade.fees,
            'pnl': trade.pnl,
        }
        for trade in result.trades
    ]

print('回測、ledger 與 metrics 函式已載入')

## 6. Deterministic synthetic smoke fixture 與可選 CSV upload

synthetic fixture 故意放入少量可辨識的上升、回檔、2% 以上反彈與量能脈衝，讓回測通常能產生交易；它不是市場資料，也不代表 Study evidence。

In [ ]:
def make_synthetic_fixture(rows: int = 420, seed: int = SMOKE_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range('2022-01-03', periods=rows)
    position = np.arange(rows, dtype=float)
    close = 100.0 * np.exp(0.0008 * position + 0.004 * np.sin(position / 9.0))
    close *= 1.0 + rng.normal(0.0, 0.0015, rows)
    volume = 1_000_000.0 * (1.0 + 0.05 * np.sin(position / 11.0))
    signal_indices = list(range(60, rows - 25, 40))
    for number, signal_index in enumerate(signal_indices):
        anchor = close[signal_index - 6]
        close[signal_index - 6: signal_index + 1] = [
            anchor,
            anchor * 1.005,
            anchor * 1.010,
            anchor * 1.015,
            anchor * 1.020,
            anchor * 1.020 * 0.980,
            anchor * 1.020 * 0.980 * 1.040,
        ]
        volume[signal_index - 6: signal_index - 2] = 3_000_000.0
        volume[signal_index - 2] = 1_000_000.0
        if number % 3 == 1:
            close[signal_index + 1: signal_index + 4] = [
                close[signal_index] * 1.010,
                close[signal_index] * 1.025,
                close[signal_index] * 1.050,
            ]
        elif number % 3 == 2:
            close[signal_index + 1: signal_index + 4] = [
                close[signal_index] * 0.995,
                close[signal_index] * 0.980,
                close[signal_index] * 0.955,
            ]
    gaps = 1.0 + 0.0002 * np.sin(position / 5.0)
    open_price = close.copy()
    open_price[1:] = close[:-1] * gaps[1:]
    high = np.maximum(open_price, close) * 1.003
    low = np.minimum(open_price, close) * 0.997
    return pd.DataFrame(
        {'Open': open_price, 'High': high, 'Low': low, 'Close': close, 'Volume': volume},
        index=dates,
    )


def read_csv_from_colab_upload() -> pd.DataFrame:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError('USE_UPLOADED_DATA=True 時，請在 Google Colab 執行 upload cell') from exc
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('沒有上傳 CSV；請重新執行此 cell 或改回 USE_UPLOADED_DATA=False')
    filename = UPLOADED_FILENAME or next(iter(uploaded))
    if filename not in uploaded:
        raise ValueError('找不到指定的 UPLOADED_FILENAME: ' + str(filename))
    return prepare_ohlcv(pd.read_csv(filename))


if USE_UPLOADED_DATA:
    bars = read_csv_from_colab_upload()
    data_source = 'Colab uploaded CSV'
else:
    bars = validate_ohlcv(make_synthetic_fixture())
    data_source = 'deterministic synthetic smoke fixture'

print('資料來源：', data_source)
print('sessions：', len(bars), '| 起訖：', bars.index.min().date(), '→', bars.index.max().date())
print(bars.head(3).to_string())

## 7. Look-ahead 防護的 deterministic test

測試會改動一個訊號日當根的 `Open`、`High`、`Low`、`Volume`，再比較該日的 forward-response 欄位與 `raw_signal`。`Close` 刻意不改，因為 Study 的當日價格加速、RSI、收盤方向本來就把已完成的訊號日 Close 當成價格確認輸入；這個測試隔離的是「延遲回應不可讀取訊號日以後資料」的 look-ahead 風險，而不是宣稱任何當日價格條件都與當日 Close 無關。

In [ ]:
def run_lookahead_test() -> None:
    sample = make_synthetic_fixture(rows=180, seed=SMOKE_SEED)
    signal_index = 60
    signal_session = sample.index[signal_index]
    changed = sample.copy()
    changed.loc[signal_session, 'Open'] = sample.loc[signal_session, 'Open'] * 1.25
    changed.loc[signal_session, 'High'] = sample.loc[signal_session, 'Open'] * 1.25
    changed.loc[signal_session, 'Low'] = sample.loc[signal_session, 'Open'] * 0.75
    changed.loc[signal_session, 'Volume'] = sample.loc[signal_session, 'Volume'] * 99.0
    before = indicators(sample).loc[signal_session]
    after = indicators(changed).loc[signal_session]
    numeric_columns = [
        'prior_volume_ratio',
        'prior_volume_forward_return',
        'prior_unweighted_forward_return',
        'prior_volume_forward_advantage',
    ]
    for column in numeric_columns:
        np.testing.assert_allclose(before[column], after[column], equal_nan=True)
    for column in ['volume_forward_response_lead', 'raw_signal']:
        assert bool(before[column]) == bool(after[column]), column
    print('PASS：改動訊號日 Open/High/Low/Volume 後，forward-response 與 raw_signal 不變。')


run_lookahead_test()

## 8. 查看 candidate / baseline 的訊號欄位

這一格只列出資料本身算出的欄位，不嵌入既有 Development evidence 數值。

In [ ]:
candidate_indicators = indicators(bars, DEFAULT_SPEC)
baseline_indicators = indicators(bars, BASELINE_SPEC)
candidate_signals = candidate_indicators.loc[candidate_indicators['raw_signal']]
baseline_signals = baseline_indicators.loc[baseline_indicators['raw_signal']]
signal_columns = [
    'Close', 'Volume', 'sma_20', 'rsi_2', 'trend_slope_ratio',
    'price_acceleration', 'prior_volume_ratio',
    'prior_volume_forward_return', 'prior_unweighted_forward_return',
    'prior_volume_forward_advantage', 'raw_signal',
]
print('candidate raw signals:', len(candidate_signals))
print(candidate_signals[signal_columns].head(12).to_string())
print('\nbaseline raw signals:', len(baseline_signals))
print(baseline_signals[signal_columns].head(12).to_string())

## 9. 離線回測：base / stress、交易 ledger 與摘要 metrics

每個模型都在相同的 bars、日期範圍與執行規則下各跑一次 base 與 stress。summary 的 return 是完成交易 PnL 除以初始資金；profit factor 是總獲利除以總虧損；maximum drawdown 依完成交易後的 realized equity 計算。這些數字只屬於本次 notebook 輸入。

In [ ]:
specs = {'candidate': DEFAULT_SPEC, 'baseline': BASELINE_SPEC}
costs = {'base': BASE_COST, 'stress': STRESS_COST}
results: dict[tuple[str, str], BacktestResult] = {}
summary_rows: list[dict[str, object]] = []
ledger_rows: list[dict[str, object]] = []

for model_name, spec in specs.items():
    for scenario_name, cost in costs.items():
        result = backtest(
            bars,
            spec=spec,
            cost=cost,
            signal_start=SIGNAL_START,
            signal_end=SIGNAL_END,
        )
        results[(model_name, scenario_name)] = result
        metrics = qualification_metrics(result, initial_cash=INITIAL_CASH)
        summary_rows.append({'model': model_name, 'scenario': scenario_name, **metrics})
        ledger_rows.extend(ledger_for_result(model_name, scenario_name, result))

summary = pd.DataFrame(summary_rows)
summary = summary[['model', 'scenario', 'completed_trades', 'traded_years', 'return', 'profit_factor', 'maximum_drawdown']]
print(summary.to_string(index=False, formatters={
    'return': lambda value: f'{value:.4%}',
    'profit_factor': lambda value: 'inf' if math.isinf(value) else f'{value:.4f}',
    'maximum_drawdown': lambda value: f'{value:.4%}',
}))

ledger = pd.DataFrame(ledger_rows)
if ledger.empty:
    print('本次輸入沒有完成交易；ledger 為空。')
else:
    print('\n交易 ledger：')
    print(ledger.to_string(index=False))

## 10. Equity / trade summary 圖

左圖將每筆完成交易的 realized equity 依退出日連接；右圖比較四種模型—成本組合的交易數。沒有交易時仍會畫出初始資金線，方便辨識是程式成功但訊號稀疏，還是 cell 失敗。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colours = {
    ('candidate', 'base'): '#1f77b4',
    ('candidate', 'stress'): '#6baed6',
    ('baseline', 'base'): '#d62728',
    ('baseline', 'stress'): '#ff9896',
}
for key, result in results.items():
    curve = equity_curve(result)
    axes[0].step(
        curve['Date'], curve['Equity'], where='post',
        label=f'{key[0]} / {key[1]}', color=colours[key],
    )
axes[0].axhline(INITIAL_CASH, color='black', linewidth=0.8, alpha=0.5)
axes[0].set_title('Realized equity（依退出日）')
axes[0].set_ylabel('資產')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.25)

labels = [f"{row['model']} / {row['scenario']}" for _, row in summary.iterrows()]
counts = summary['completed_trades'].to_numpy()
bar_colours = [colours[(row['model'], row['scenario'])] for _, row in summary.iterrows()]
axes[1].bar(labels, counts, color=bar_colours)
axes[1].set_title('Completed trades')
axes[1].set_ylabel('交易數')
axes[1].tick_params(axis='x', rotation=35)
axes[1].grid(axis='y', alpha=0.25)
fig.suptitle(f'TSM response-lag notebook smoke / {data_source}', y=1.02)
fig.tight_layout()
plt.show()

## 11. 重現方式與限制

1. 不上傳資料時，直接從頭執行所有 cells；預設會完成 deterministic synthetic smoke。
2. 在 Colab 將 `USE_UPLOADED_DATA=True`，重新執行資料切換 cell，選擇包含 `Date` 或 `Datetime`、`Open`、`High`、`Low`、`Close`、`Volume` 的 CSV，再從 look-ahead test 往下執行。
3. 如需限定訊號期間，可在設定 cell 填入 `SIGNAL_START` 與 `SIGNAL_END`；這是 notebook 輸入設定，不是對正式 Study 的事後調參。

上傳資料必須已經是 XNYS session-clean、Yahoo auto-adjusted 的日線 OHLCV；本 notebook 不會補抓假日、修正供應商資料、下載外部資料或把重跑結果寫回 Study。日線 OHLCV 也無法辨識盤中主動買賣方向；`d→d+1` 配對是描述性訊號條件，不是因果證明。最重要的是，這份 Colab reproducer 不含正式 Evaluation／Terminal 結果，也不能把 smoke 或任意上傳資料的輸出宣稱為正式 evidence。